### Libraries

In [1]:
import re
import os
import numpy as np
from astropy.io import ascii
from astropy.io import fits
from astropy.table import Table
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from io import StringIO
from matplotlib.ticker import AutoMinorLocator
from astropy.constants import L_sun
from matplotlib import gridspec
import astropy.units as u
from math import pi
import gc
from matplotlib.ticker import ScalarFormatter
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from astropy.table import Column
from scipy.stats import bootstrap

from __future__ import annotations
import os
import json
from astropy.cosmology import FlatwCDM

import pymultinest
import corner
from getdist import plots, MCSamples
import scipy.optimize as op
from scipy import stats
from scipy.stats import norm
from astropy.time import Time
import matplotlib.dates as mdates

In [2]:
def HyperLeda(filename):
    """
    Lee una tabla tipo NED/HyperLEDA ignorando comentarios y headers,
    devolviendo todas las columnas en un DataFrame de pandas.
    """

    # Leer el archivo ignorando líneas de comentario
    df = pd.read_csv(
        filename,
        comment="#",
        sep=",",
        engine="python"
    )

    # Limpiar nombres de columnas
    df.columns = [c.strip() for c in df.columns]

    #Convertir todo lo que se pueda a numérico
    #for col in df.columns:
        #df[col] = pd.to_numeric(df[col], errors="ignore")

    return df

def SIMBAD(filename):
    """
    Lee una tabla tipo NED/HyperLEDA ignorando comentarios y headers,
    devolviendo todas las columnas en un DataFrame de pandas.
    """

    # Leer el archivo ignorando líneas de comentario
    df = pd.read_csv(
        filename,
        comment="#",
        sep="|",
        engine="python"
    )

    # Limpiar nombres de columnas
    df.columns = [c.strip() for c in df.columns]

    #Convertir todo lo que se pueda a numérico
    #for col in df.columns:
        #df[col] = pd.to_numeric(df[col], errors="ignore")

    return df

def NED(filename):
    """
    Lee una tabla tipo NED/HyperLEDA ignorando comentarios y headers,
    devolviendo todas las columnas en un DataFrame de pandas.
    """

    # Leer el archivo ignorando líneas de comentario
    df = pd.read_csv(
        filename,
        comment="#",
        sep=",",
        engine="python"
    )

    # Limpiar nombres de columnas
    df.columns = [c.strip() for c in df.columns]

    #Convertir todo lo que se pueda a numérico
    #for col in df.columns:
        #df[col] = pd.to_numeric(df[col], errors="ignore")

    return df

In [3]:
df_dist = NED("/home/holman/HIIGalaxies/Modulus/ref_NED/1.IC0010.csv")

PLRC = np.unique(df_dist[df_dist['Method']=='Cepheids']['Reference'])
TRGB = np.unique(df_dist[df_dist['Method']=='TRGB']['Reference'])
RRLy = np.unique(df_dist[df_dist['Method']=='RR Lyrae']['Reference'])

print(len(PLRC), len(TRGB), len(RRLy))

1 9 1


In [4]:
home = '/home/holman/HIIGalaxies/Modulus/ref_NED'
list_files = os.listdir(home)
list_files.sort(key=lambda x: int(x.split('.')[0]))

for b in list_files:
    df_dist = NED(f"/home/holman/HIIGalaxies/Modulus/ref_NED/{b}")
    PLRC = np.unique(df_dist[df_dist['Method']=='Cepheids']['Reference'])
    TRGB = np.unique(df_dist[df_dist['Method']=='TRGB']['Reference'])
    RRLy = np.unique(df_dist[df_dist['Method']=='RR Lyrae']['Reference'])

    print(b,"     ",len(PLRC),"	", len(TRGB),"	", len(RRLy),"\n")


0.LMC.csv       64 	 8 	 40 

1.IC0010.csv       1 	 9 	 1 

2.IC2574.csv       0 	 4 	 0 

3.M33.csv       16 	 17 	 6 

4.M81.csv       12 	 8 	 0 

5.M101.csv       14 	 7 	 0 

6.MRK116.csv       3 	 3 	 0 

7.NGC0925.csv       10 	 0 	 0 

8.NGC2366.csv       1 	 7 	 0 

9.NGC2403.csv       4 	 5 	 0 

10.NGC2541.csv       8 	 0 	 0 

11.NGC3198.csv       11 	 0 	 0 

12.NGC3319.csv       9 	 0 	 0 

13.NGC4214.csv       0 	 8 	 0 

14.NGC4236.csv       0 	 3 	 0 

15.NGC4258.csv       25 	 6 	 0 

16.NGC4395.csv       1 	 4 	 0 

17.NGC6822.csv       14 	 9 	 3 

18.NGC1073.csv       0 	 0 	 0 

19.NGC2500.csv       0 	 1 	 0 

20.NGC3184.csv       0 	 0 	 0 

21.M96.csv       11 	 4 	 0 

22.NGC3370.csv       8 	 1 	 0 

23.M66.csv       12 	 4 	 0 

24.NGC4414.csv       10 	 0 	 0 

25.NGC4496.csv       0 	 0 	 0 

26.NGC4535.csv       10 	 0 	 0 

27.NGC4536.csv       20 	 0 	 0 

28.NGC4725.csv       9 	 0 	 0 

29.UGC08091.csv       1 	 4 	 0 

30.NGC5204.csv       0 	 2 	 0

In [21]:
home = '/home/holman/HIIGalaxies/Modulus/ref_HyperLeda'
list_files = os.listdir(home)
list_files.sort(key=lambda x: int(x.split('.')[0]))

for b in list_files:
    df_dist = HyperLeda(f"/home/holman/HIIGalaxies/Modulus/ref_HyperLeda/{b}")
    df_dist = df_dist[df_dist['bibcode'].str[0:4].astype(int) >= 2000]
    df_dist['$link[method_method]'] = df_dist['$link[method_method]'].str.strip()
    df_dist['bibcode'] = df_dist['bibcode'].str.strip()

    PLRC = np.unique(df_dist[df_dist['$link[method_method]']=='Cepheids']['bibcode'])
    TRGB = np.unique(df_dist[df_dist['$link[method_method]']=='TRGB']['bibcode'])
    RRLy = np.unique(df_dist[df_dist['$link[method_method]']=='RRLyrae']['bibcode'])

    print(b,"     ",len(PLRC),"	", len(TRGB),"	", len(RRLy),"\n")

0.LMC.csv       13 	 3 	 4 

1.IC0010.csv       3 	 2 	 0 

2.IC2574.csv       0 	 6 	 0 

3.M33.csv       11 	 12 	 3 

4.M81.csv       9 	 9 	 0 

5.M101.csv       10 	 6 	 0 

6.MRK116.csv       2 	 2 	 0 

7.NGC0925.csv       9 	 1 	 0 

8.NGC2366.csv       0 	 6 	 0 

9.NGC2403.csv       4 	 5 	 0 

10.NGC2541.csv       7 	 1 	 0 

11.NGC3198.csv       8 	 0 	 0 

12.NGC3319.csv       8 	 0 	 0 

13.NGC4214.csv       0 	 10 	 0 

14.NGC4236.csv       0 	 5 	 0 

15.NGC4258.csv       15 	 7 	 0 

16.NGC4395.csv       2 	 6 	 0 

17.NGC6822.csv       10 	 6 	 2 

18.NGC1073.csv       0 	 0 	 0 

19.NGC2500.csv       0 	 0 	 0 

20.NGC3184.csv       0 	 0 	 0 

21.M96.csv       9 	 3 	 0 

22.NGC3370.csv       2 	 0 	 0 

23.M66.csv       12 	 3 	 0 

24.NGC4414.csv       10 	 0 	 0 

25.NGC4496.csv       11 	 0 	 0 

26.NGC4535.csv       9 	 0 	 0 

27.NGC4536.csv       11 	 0 	 0 

28.NGC4725.csv       8 	 0 	 0 

29.UGC08091.csv       0 	 6 	 0 

30.NGC5204.csv       0 	 6 	 0 

3

In [23]:
home = '/home/holman/HIIGalaxies/Modulus/ref_SIMBAD'
list_files = os.listdir(home)
list_files.sort(key=lambda x: int(x.split('.')[0]))

for b in list_files:
    df_dist = SIMBAD(f"/home/holman/HIIGalaxies/Modulus/ref_SIMBAD/{b}")
    df_dist = df_dist[df_dist['reference'].str[0:4].astype(int) >= 2000]
    #df_dist['$link[method_method]'] = df_dist['$link[method_method]'].str.strip()
    #df_dist['bibcode'] = df_dist['bibcode'].str.strip()

    #PLRC = np.unique(df_dist[df_dist['$link[method_method]']=='Cepheids']['bibcode'])
    #TRGB = np.unique(df_dist[df_dist['$link[method_method]']=='TRGB']['bibcode'])
    #RRLy = np.unique(df_dist[df_dist['$link[method_method]']=='RRLyrae']['bibcode'])

    #print(b,"     ",len(PLRC),"	", len(TRGB),"	", len(RRLy),"\n")

df_dist

,distance,distance Q unit,err- err+,method,reference,Unnamed: 5
0,distance,18.4 Mpc,-5.2 +5.2,T-F,2022MNRAS.511.6160K,NaN
1,distance,20.04 Mpc,-3.88 +3.88,T-F,2020ApJ...902..145K,NaN
2,distance,28.1838 Mpc,,redshift,2019A&A...631A..38L,NaN
3,distance,22.08 Mpc,,,2016AJ....152...50T,NaN
4,distance,23.1 Mpc,-4.80 +4.80,T-F,2011PASP..123.1011A,NaN
5,distance,18.0 Mpc,,T-F,2014MNRAS.444..527S,NaN
6,distance,22.49 Mpc,-0.07 +0.07,,2013AJ....146...86T,NaN
7,distance,26.0 Mpc,,redshift,2011MNRAS.413..813C,NaN
8,distance,26.03 Mpc,,redshift,2007ApJ...655..790C,NaN
